<a href="https://colab.research.google.com/github/halilkrkn/natural-gas-anomaly-detection/blob/main/notebooks/01_phase3_environment_and_weymouth_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Faz 3 — Colab ortamı ve Weymouth sayısal doğrulaması

**Amaç:** GitHub deposundan kodu alıp Colab ortamında bağımlılıkları kurmak, `pip install -e .` ve temel testleri çalıştırmak, ardından gerçek `fluids.compressible.Weymouth` çağrısıyla sayısal smoke testlerini yürütmek.

**Önemli sınırlar**
- Bu notebook model eğitimi veya nihai deney değildir.
- Paket konfigürasyonundaki bir sayı bilimsel kararın kapandığını göstermez.
- Başarılı API/geri çözüm smoke testleri, seçilen sabit `Zavg=0.8065` varsayımının fiziksel yeterliliğini veya sızıntı modelinin fiziksel yeterliliğini tek başına doğrulamaz. `OPEN-DECISION-012`, ilgili kanıtlar karar defterine işlenene kadar açık kalır.


## 2. Google Drive kalıcı çıktı klasörü

Colab sanal makinesindeki `/content` kalıcı değildir. Doğrulama raporlarını ve ortam manifestini Drive'a yazacağız. Proje kodu GitHub'ın kaynak kontrolünde kalacak; büyük veri/checkpoint dosyaları Git'e eklenmeyecek.

In [35]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/natural-gas-anomaly-detection")
(DRIVE_ROOT / "reports" / "metrics").mkdir(parents=True, exist_ok=True)
print("Persistent report directory:", DRIVE_ROOT / "reports" / "metrics")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Persistent report directory: /content/drive/MyDrive/natural-gas-anomaly-detection/reports/metrics


## 3. Bağımlılık kurulumu ve editable package

`fluids` için doğrulama amacıyla sürüm `1.3.1` sabitlenmiştir. Diğer bağımlılıklar bu ilk ortam smoke testinde başlangıç listesi olarak kurulur; nihai deneylerden önce Colab'da gerçekten çalışan sürümlerin manifesti kaydedilecektir.

**Colab uyarısı:** `pip install` sonrasında çalışma zamanı yeniden başlarsa notebook'u baştan çalıştır; önce depo ayarı/klonlama, ardından bağımlılık kurulum hücrelerinin tamamlandığını doğrula. Sürüm sabitlemesi ancak manifestte gerçekten çalışan sürümler görüldükten sonra yapılacaktır.


In [36]:
%cd /content/natural-gas-anomaly-detection
%pip install -r requirements.txt
%pip install -e .

/content/natural-gas-anomaly-detection
Obtaining file:///content/natural-gas-anomaly-detection
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for natural-gas-anomaly-detection (pyproject.toml) ... done
  Created wheel for natural-gas-anomaly-detection: filename=natural_gas_anomaly_detection-0.1.0-0.editable-py3-none-any.whl size=1648 sha256=2cc8132d5d3d9da719b9658115d9106a9f9770a3f76f42198e9a64fb2b9aa2d9
  Stored in directory: /tmp/pip-ephem-wheel-cache-3ehmolqz/wheels/42/e1/e5/697e6c943fa4cb08c1c29bf190f53b6a606dca075c900e04ad
Successfully built natural-gas-anomaly-detection
  Attempting uninstall: natural-gas-anomaly-detection
    Found existing installation: natural-gas-anomaly-detection 0.1.0
    Uninstalling natural-gas-anomaly-detection-0.1.0:
      Successfully uninstalled natural-gas-anomaly-detectio

In [37]:
import fluids
from fluids.compressible import Weymouth
import inspect
print("fluids version:", getattr(fluids, "__version__", "unknown"))
print("Weymouth signature:", inspect.signature(Weymouth))
print("Weymouth module:", Weymouth.__module__)

fluids version: 1.3.1
Weymouth signature: (SG: 'float', Tavg: 'float', L: 'float | None' = None, D: 'float | None' = None, P1: 'float | None' = None, P2: 'float | None' = None, Q: 'float | None' = None, Ts: 'float' = 288.7, Ps: 'float' = 101325.0, Zavg: 'float' = 1.0, E: 'float' = 0.92) -> 'float'
Weymouth module: fluids.compressible


## 4. Temel testler

Bu testler depo yapısı ve Weymouth gerçek kütüphane çağrısı dâhil temel kontrol içindir. `test_weymouth_validation.py` Colab'da **skip olmadan çalışmalıdır**. Atlanan veya başarısız test, Faz 3'ün geçtiği anlamına gelmez.

In [38]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=PROJECT_ROOT, text=True, capture_output=True,
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print("pytest exit code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("pytest başarısız oldu. Hataları düzeltmeden sonraki adıma geçme.")

..                                                                       [100%]
2 passed in 0.28s

pytest exit code: 0


## 5. Weymouth sayısal doğrulama

Kontroller:
1. `fluids` dokümantasyonundaki örnek sonucu yeniden üret.
2. Proje nominal noktasıyla önce `Q0` hesapla, sonra aynı akışı kullanarak `P2` değerini geri çöz.
3. Talep akışını ±%5 değiştirince aşağı akım basıncının sonlu/pozitif kalmasını ve akış artarken basıncın düşmesini doğrula.
4. Orta noktada sızıntı için `Qin = Qout + Qleak` kütle dengesini ve iki segmentin basınç sıralamasını kontrol et. v1 için sabit `Zavg=0.8065` seçilmiştir; smoke test tek başına OPEN-DECISION-012 sayısal kapanışını sağlamaz.

In [39]:
from pathlib import Path
import json

OUTPUT_JSON = DRIVE_ROOT / "reports" / "metrics" / "weymouth_validation.json"
run = subprocess.run(
    [sys.executable, "scripts/run_weymouth_validation.py", "--output", str(OUTPUT_JSON)],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
print(run.stdout)
if run.stderr:
    print("STDERR:", run.stderr)
if run.returncode != 0:
    raise RuntimeError("Weymouth validation başarısız. OPEN-DECISION-012 kapatılmamalı.")

report = json.loads(OUTPUT_JSON.read_text(encoding="utf-8"))
assert report["all_passed"] is True
print("Report persisted to:", OUTPUT_JSON)

{
  "fluids_version": "1.3.1",
  "baseline_parameters": {
    "SG": 0.675,
    "Tavg": 283.15,
    "L": 100000.0,
    "D": 0.75,
    "P1": 9000000.0,
    "P2": 5000000.0,
    "Ts": 288.15,
    "Ps": 101325.0,
    "Zavg": 0.8065,
    "E": 0.92
  },
  "checks": [
    {
      "name": "fluids_documented_example",
      "Q_m3_s": 32.07729055913029,
      "expected_Q_m3_s": 32.07729055913029,
      "relative_error": 0.0,
      "passed": true
    },
    {
      "name": "nominal_pressure_flow_round_trip",
      "Q0_m3_s_at_Ts_Ps": 317.9610932418773,
      "P2_requested_pa": 5000000.0,
      "P2_recovered_pa": 4999999.999999998,
      "relative_pressure_error": 3.725290298461914e-16,
      "passed": true
    },
    {
      "name": "plus_minus_5_percent_demand_bounds",
      "demand_multiplier": [
        0.95,
        1.0,
        1.05
      ],
      "P2_pa": [
        5519057.890618651,
        4999999.999999998,
        4388621.651498333
      ],
      "passed": true,
      "interpretation": 

## 6. Ortam manifesti

Python/platform ve temel paket sürümlerini kaydet. Bu dosyayı nihai deneylerde de saklayacağız.

In [40]:
manifest_cmd = subprocess.run(
    [sys.executable, "scripts/capture_environment.py"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
    check=True,
)
print(manifest_cmd.stdout)
local_manifest = PROJECT_ROOT / "reports" / "metrics" / "phase3_environment_manifest.json"
persistent_manifest = DRIVE_ROOT / "reports" / "metrics" / "phase3_environment_manifest.json"
persistent_manifest.write_text(local_manifest.read_text(encoding="utf-8"), encoding="utf-8")
print("Persistent manifest:", persistent_manifest)

{
  "timestamp_utc": "2026-10-16T23:14:00.759340+00:00",
  "python_version": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "packages": {
    "numpy": "2.1.3",
    "pandas": "2.2.3",
    "scipy": "1.16.3",
    "scikit-learn": "1.6.1",
    "torch": "2.11.0+cu130",
    "fluids": "1.3.1",
    "PyYAML": "6.0.3",
    "pytest": "8.4.2"
  },
  "torch_runtime": {
    "available": true,
    "version": "2.11.0+cu130",
    "cuda_available": true,
    "cuda_version": "13.0",
    "device_count": 1
  }
}
Manifest written to: /content/natural-gas-anomaly-detection/reports/metrics/phase3_environment_manifest.json

Persistent manifest: /content/drive/MyDrive/natural-gas-anomaly-detection/reports/metrics/phase3_environment_manifest.json


## 7. Faz 3 durumunu kaydet

Bu notebook bittiğinde şunları karar defterine ekle:
- GitHub repo URL'si, ilk commit SHA'sı ve ana dal adı.
- Colab çalışma tarihi/ortamı, Python ve `fluids` sürümü.
- `pip install -r requirements.txt`, `pip install -e .` ve `pytest -q` sonuçları.
- `weymouth_validation.json` içindeki her kontrolün çıktısı.

**Bilimsel kapanış kuralı:** Testler geçerse yalnızca kütüphane API'si, nominal geri çözüm ve temel sayısal davranış doğrulanmış olur. `OPEN-DECISION-012` ancak seçilen sabit `Zavg=0.8065` varsayımının sınırlılıkları ve sade sızıntı modelinin yeterliliği ayrıca gözden geçirilip kanıt/karar kaydı/dosya güncellemeleri tamamlanınca kapanabilir. Bu notebook tek başına kararı otomatik kapatmaz.

In [41]:
!pytest -q

..                                                                       [100%]
2 passed in 0.12s
